<a href="https://colab.research.google.com/github/dharshini-dev-hub/bigdata-lab/blob/rdd-operations/Rdd_aggregations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark Connection

In [2]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,853 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,773 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-s

In [3]:
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz

In [4]:
!tar xf spark-3.4.1-bin-hadoop3.tgz

In [5]:
import os
os.environ["JAVA_HOME"] = f"/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/spark-3.4.1-bin-hadoop3"

In [6]:
!pip install -q pyspark==3.5.1

In [7]:
!pip install findspark

In [8]:
import findspark
findspark.init()

In [9]:
import pyspark

# RDD Practice problems

In [10]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("RDD_operation_2") \
.getOrCreate()

In [11]:
rdd = spark.sparkContext.textFile("/content/sample_data/customers.csv")
rdd.take(5)

['customer_id,name,city,state,country,registration_date,is_active',
 '0,Customer_0,Pune,Maharashtra,India,2023-06-29,False',
 '1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True',
 '2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True',
 '3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False']

In [12]:
header = rdd.first()
header

'customer_id,name,city,state,country,registration_date,is_active'

In [13]:
customers_rdd = rdd.filter(lambda x: x!=header)
customers_rdd.take(5)

['0,Customer_0,Pune,Maharashtra,India,2023-06-29,False',
 '1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True',
 '2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True',
 '3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False',
 '4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,False']

In [14]:
customers = customers_rdd.flatMap(lambda x : x.split(','))
customers.take(5)

['0', 'Customer_0', 'Pune', 'Maharashtra', 'India']

In [15]:
numbers = spark.sparkContext.parallelize([1,2,3,4,5])
squared_numbers = numbers.map(lambda x:x**2)
squared_numbers.collect()

[1, 4, 9, 16, 25]

In [16]:
even_numbers = squared_numbers.filter(lambda x:x%2==0)
even_numbers.collect()

[4, 16]

In [17]:
sentences = [
    "Spark is fast",
    "It is built on Hadoop",
    "Big data processing is easy with Spark"
]

In [18]:
sentence = spark.sparkContext.parallelize(sentences)
sentence.collect()

['Spark is fast',
 'It is built on Hadoop',
 'Big data processing is easy with Spark']

In [19]:
map_sentence = sentence.map(lambda x:x.split())
map_sentence.collect()

[['Spark', 'is', 'fast'],
 ['It', 'is', 'built', 'on', 'Hadoop'],
 ['Big', 'data', 'processing', 'is', 'easy', 'with', 'Spark']]

In [20]:
flatmap_sentence = sentence.flatMap(lambda x:x.split())
flatmap_sentence.collect()

['Spark',
 'is',
 'fast',
 'It',
 'is',
 'built',
 'on',
 'Hadoop',
 'Big',
 'data',
 'processing',
 'is',
 'easy',
 'with',
 'Spark']

In [21]:
duplicates = flatmap_sentence.distinct()
duplicates.collect()

['fast',
 'It',
 'built',
 'Hadoop',
 'Big',
 'easy',
 'with',
 'Spark',
 'is',
 'on',
 'data',
 'processing']

In [22]:
numbers = spark.sparkContext.parallelize([6,7,8,9,10])
numbers.count()

5

In [23]:
sum_numbers = spark.sparkContext.parallelize([11,12,13,14,15])
sum_numbers.reduce(lambda x, y: x+y)

65

In [24]:
max_numbers = spark.sparkContext.parallelize([17,16])
max_numbers.reduce(lambda x,y: x if x>y else y)

17

In [25]:
sentences = [
    "Spark is fast",
    "It is built on Hadoop",
    "Big data processing is easy with Spark"
]

In [26]:
sentence = spark.sparkContext.parallelize(sentences)
sentence.collect()

['Spark is fast',
 'It is built on Hadoop',
 'Big data processing is easy with Spark']

In [27]:
word_count = sentence.flatMap(lambda x:x.split()).map(lambda x:(x,1)).reduceByKey(lambda x,y:x+y)
word_count.collect()

[('fast', 1),
 ('It', 1),
 ('built', 1),
 ('Hadoop', 1),
 ('Big', 1),
 ('easy', 1),
 ('with', 1),
 ('Spark', 2),
 ('is', 3),
 ('on', 1),
 ('data', 1),
 ('processing', 1)]

In [28]:
sentence.flatMap(lambda x:x.split()).countByValue()

defaultdict(int,
            {'Spark': 2,
             'is': 3,
             'fast': 1,
             'It': 1,
             'built': 1,
             'on': 1,
             'Hadoop': 1,
             'Big': 1,
             'data': 1,
             'processing': 1,
             'easy': 1,
             'with': 1})

In [29]:
data = [
    ("HR", 40000),
    ("IT", 60000),
    ("HR", 42000),
    ("IT", 75000),
    ("Finance", 50000),
    ("IT", 62000),
    ("Finance", 55000),
    ("HR", 39000)
]

In [30]:
data_rdd = spark.sparkContext.parallelize(data)
data_rdd.collect()

[('HR', 40000),
 ('IT', 60000),
 ('HR', 42000),
 ('IT', 75000),
 ('Finance', 50000),
 ('IT', 62000),
 ('Finance', 55000),
 ('HR', 39000)]

In [31]:
group_rdd = data_rdd.groupByKey()
grouped_salaries = group_rdd.mapValues(list)
grouped_salaries.collect()

[('HR', [40000, 42000, 39000]),
 ('IT', [60000, 75000, 62000]),
 ('Finance', [50000, 55000])]

In [32]:
avg_salaries = grouped_salaries.mapValues(lambda y: sum(y)/len(y))
avg_salaries.collect()

[('HR', 40333.333333333336), ('IT', 65666.66666666667), ('Finance', 52500.0)]

In [33]:
pair_rdd = data_rdd.map(lambda x: (x[0], (x[1], 1)))
pair_rdd.collect()

[('HR', (40000, 1)),
 ('IT', (60000, 1)),
 ('HR', (42000, 1)),
 ('IT', (75000, 1)),
 ('Finance', (50000, 1)),
 ('IT', (62000, 1)),
 ('Finance', (55000, 1)),
 ('HR', (39000, 1))]

In [34]:
sum_count_rdd = pair_rdd.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
sum_count_rdd.collect()

[('HR', (121000, 3)), ('IT', (197000, 3)), ('Finance', (105000, 2))]

In [35]:
avg_salaries = sum_count_rdd.mapValues(lambda x: x[0]/x[1])
avg_salaries.collect()

[('HR', 40333.333333333336), ('IT', 65666.66666666667), ('Finance', 52500.0)]

In [36]:
group_rdd = data_rdd.groupByKey()
max_salary = group_rdd.mapValues(lambda x: max(x))
max_salary.collect()

[('HR', 42000), ('IT', 75000), ('Finance', 55000)]

In [37]:
grouped_rdd = data_rdd.groupByKey()
len_employees = grouped_rdd.mapValues(lambda x: len(x))
len_employees.collect()

[('HR', 3), ('IT', 3), ('Finance', 2)]

In [38]:
avg_salary = grouped_rdd.mapValues(lambda x: sum(x)/len(x))
avg_salary.collect()

[('HR', 40333.333333333336), ('IT', 65666.66666666667), ('Finance', 52500.0)]

In [39]:
max_avg_salary = avg_salary.filter(lambda x: x[1]>60000)
max_avg_salary.collect()

[('IT', 65666.66666666667)]

In [40]:
total_salary = data_rdd.reduceByKey(lambda x,y: x+y)
total_salary.collect()

[('HR', 121000), ('IT', 197000), ('Finance', 105000)]

In [41]:
grouped_rdd = data_rdd.reduceByKey(lambda x,y:max(x,y))
grouped_rdd.collect()

[('HR', 42000), ('IT', 75000), ('Finance', 55000)]

# ReduceByKey

In [42]:
pair_rdd = data_rdd.map(lambda x: (x[0], (x[1], 1)))
pair_rdd.collect()

[('HR', (40000, 1)),
 ('IT', (60000, 1)),
 ('HR', (42000, 1)),
 ('IT', (75000, 1)),
 ('Finance', (50000, 1)),
 ('IT', (62000, 1)),
 ('Finance', (55000, 1)),
 ('HR', (39000, 1))]

In [43]:
total_salary = pair_rdd.reduceByKey(lambda x,y: (x[0]+y[0],x[1]+y[1]))
total_salary.collect()

[('HR', (121000, 3)), ('IT', (197000, 3)), ('Finance', (105000, 2))]

In [44]:
average_salary = total_salary.mapValues(lambda x: x[0]/x[1])
average_salary.collect()

[('HR', 40333.333333333336), ('IT', 65666.66666666667), ('Finance', 52500.0)]

In [45]:
customers = [("Alice", 200), ("Bob", 100), ("Alice", 300), ("Bob", 400), ("Alice", 500)]

In [46]:
customers_rdd = spark.sparkContext.parallelize(customers)
customers_rdd.collect()

[('Alice', 200), ('Bob', 100), ('Alice', 300), ('Bob', 400), ('Alice', 500)]

In [47]:
map_customers = customers_rdd.map(lambda x: (x[0],(x[1],1)))
map_customers.collect()

[('Alice', (200, 1)),
 ('Bob', (100, 1)),
 ('Alice', (300, 1)),
 ('Bob', (400, 1)),
 ('Alice', (500, 1))]

In [48]:
sum_customers = map_customers.reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1]))
sum_customers.collect()

[('Alice', (1000, 3)), ('Bob', (500, 2))]

In [49]:
avg_customers = sum_customers.mapValues(lambda x: x[0]/x[1])
avg_customers.collect()

[('Alice', 333.3333333333333), ('Bob', 250.0)]

In [50]:
marks = [("John", 85), ("Mary", 90), ("John", 78), ("Mary", 95), ("John", 82)]

In [51]:
marks_rdd = spark.sparkContext.parallelize(marks)
marks_rdd.collect()

[('John', 85), ('Mary', 90), ('John', 78), ('Mary', 95), ('John', 82)]

In [52]:
map_marks = marks_rdd.map(lambda x: (x[0],(x[1],1)))
map_marks.collect()

[('John', (85, 1)),
 ('Mary', (90, 1)),
 ('John', (78, 1)),
 ('Mary', (95, 1)),
 ('John', (82, 1))]

In [53]:
sum_marks = map_marks.reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1]))
sum_marks.collect()

[('Mary', (185, 2)), ('John', (245, 3))]

In [54]:
avg_marks = sum_marks.mapValues(lambda x: x[0]/x[1])
avg_marks.collect()

[('Mary', 92.5), ('John', 81.66666666666667)]

In [55]:
region = [("North", 12000), ("South", 10000), ("North", 15000), ("South", 13000), ("East", 11000)]

In [56]:
region_rdd = spark.sparkContext.parallelize(region)
region_rdd.collect()

[('North', 12000),
 ('South', 10000),
 ('North', 15000),
 ('South', 13000),
 ('East', 11000)]

In [57]:
map_region= region_rdd.map(lambda x: (x[0],(x[1],1)))
map_region.collect()

[('North', (12000, 1)),
 ('South', (10000, 1)),
 ('North', (15000, 1)),
 ('South', (13000, 1)),
 ('East', (11000, 1))]

In [58]:
sum_region = map_region.reduceByKey(lambda a,b :(a[0]+b[0],a[1]+b[1]))
sum_region.collect()

[('North', (27000, 2)), ('South', (23000, 2)), ('East', (11000, 1))]

In [59]:
avg_region = sum_region.mapValues(lambda x:(x[0]/x[1]))
avg_region.collect()

[('North', 13500.0), ('South', 11500.0), ('East', 11000.0)]

In [60]:
movies = [("MovieA", 4), ("MovieB", 5), ("MovieA", 3), ("MovieB", 4), ("MovieC", 5)]

In [61]:
movie_rdd = spark.sparkContext.parallelize(movies)
movie_rdd.collect()

[('MovieA', 4), ('MovieB', 5), ('MovieA', 3), ('MovieB', 4), ('MovieC', 5)]

In [62]:
map_movies = movie_rdd.map(lambda x:(x[0],(x[1],1)))
map_movies.collect()

[('MovieA', (4, 1)),
 ('MovieB', (5, 1)),
 ('MovieA', (3, 1)),
 ('MovieB', (4, 1)),
 ('MovieC', (5, 1))]

In [63]:
sum_movies = map_movies.reduceByKey(lambda a,b : (a[0]+b[0],a[1]+b[1]))
sum_movies.collect()

[('MovieB', (9, 2)), ('MovieA', (7, 2)), ('MovieC', (5, 1))]

In [64]:
avg_movies = sum_movies.mapValues(lambda x: x[0]/x[1])
avg_movies.collect()

[('MovieB', 4.5), ('MovieA', 3.5), ('MovieC', 5.0)]

In [65]:
employees = [("E1", 8), ("E2", 7), ("E1", 9), ("E2", 8), ("E3", 6)]

In [66]:
employees_rdd = spark.sparkContext.parallelize(employees)
employees_rdd.collect()

[('E1', 8), ('E2', 7), ('E1', 9), ('E2', 8), ('E3', 6)]

In [67]:
map_employees = employees_rdd.map(lambda x:(x[0],(x[1],1)))
map_employees.collect()

[('E1', (8, 1)),
 ('E2', (7, 1)),
 ('E1', (9, 1)),
 ('E2', (8, 1)),
 ('E3', (6, 1))]

In [68]:
sum_employees = map_employees.reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1]))
sum_employees.collect()

[('E3', (6, 1)), ('E1', (17, 2)), ('E2', (15, 2))]

In [69]:
avg_employees = sum_employees.mapValues(lambda x: x[0]/x[1])
avg_employees.collect()

[('E3', 6.0), ('E1', 8.5), ('E2', 7.5)]

# GroupByKey

In [70]:
purchase_gbk = [("Alice", 200), ("Bob", 100), ("Alice", 300), ("Bob", 400), ("Alice", 500)]

In [71]:
purchase_rdd = spark.sparkContext.parallelize(purchase_gbk)
purchase_rdd.collect()

[('Alice', 200), ('Bob', 100), ('Alice', 300), ('Bob', 400), ('Alice', 500)]

In [72]:
group_purchase = purchase_rdd.groupByKey()
group_purchase.collect()

[('Alice', <pyspark.resultiterable.ResultIterable at 0x79afae95d090>),
 ('Bob', <pyspark.resultiterable.ResultIterable at 0x79afae95c3d0>)]

In [73]:
group_purchase = purchase_rdd.groupByKey()
grouped_purchases_list = group_purchase.mapValues(list)
grouped_purchases_list.collect()

[('Alice', [200, 300, 500]), ('Bob', [100, 400])]

In [74]:
avg_purchase = grouped_purchases_list.mapValues(lambda x: (sum(x)/len(x)))
avg_purchase.collect()

[('Alice', 333.3333333333333), ('Bob', 250.0)]

In [75]:
group_marks = [("John", 85), ("Mary", 90), ("John", 78), ("Mary", 95), ("John", 82)]

In [76]:
group_marks_rdd = spark.sparkContext.parallelize(group_marks)
group_marks_rdd.collect()

[('John', 85), ('Mary', 90), ('John', 78), ('Mary', 95), ('John', 82)]

In [77]:
marks_group = group_marks_rdd.groupByKey()
marks_group_list = marks_group.mapValues(list)
marks_group_list.collect()

[('Mary', [90, 95]), ('John', [85, 78, 82])]

In [78]:
agg_marks_group = marks_group_list.mapValues(lambda x: (sum(x)/len(x)))
agg_marks_group.collect()

[('Mary', 92.5), ('John', 81.66666666666667)]

In [79]:
marks = [("A", 200), ("A", 400), ("C", 300), ("D", 100), ("D", 500)]

In [80]:
marks_rdd = spark.sparkContext.parallelize(marks)
max_marks = marks_rdd.reduce(lambda a,b:max(a,b))
max_marks

('D', 500)

In [81]:
top_students = marks_rdd.reduceByKey(lambda a, b: max(a, b))
top_students.collect()

[('A', 400), ('D', 500), ('C', 300)]

In [82]:
top_marks = marks_rdd.takeOrdered(3, key=lambda x: -x[1])
top_marks

[('D', 500), ('A', 400), ('C', 300)]

In [83]:
employee_data = [
    ("John", "HR", 40000),
    ("Alice", "IT", 60000),
    ("Bob", "HR", 42000),
    ("Sam", "IT", 75000),
    ("Eve", "Finance", 50000),
    ("David", "IT", 62000),
    ("Tom", "Finance", 55000),
    ("Linda", "HR", 39000)
]

In [84]:
employee_data_rdd = spark.sparkContext.parallelize(employee_data)
employee_data_rdd.collect()

[('John', 'HR', 40000),
 ('Alice', 'IT', 60000),
 ('Bob', 'HR', 42000),
 ('Sam', 'IT', 75000),
 ('Eve', 'Finance', 50000),
 ('David', 'IT', 62000),
 ('Tom', 'Finance', 55000),
 ('Linda', 'HR', 39000)]

In [85]:
employee_data_reassign = employee_data_rdd.map(lambda x: (x[1],(x[0],x[2])))
employee_data_reassign.collect()

[('HR', ('John', 40000)),
 ('IT', ('Alice', 60000)),
 ('HR', ('Bob', 42000)),
 ('IT', ('Sam', 75000)),
 ('Finance', ('Eve', 50000)),
 ('IT', ('David', 62000)),
 ('Finance', ('Tom', 55000)),
 ('HR', ('Linda', 39000))]

In [86]:
employee_salary = employee_data_reassign.reduceByKey(lambda a,b: a if a[1]>b[1] else b)
employee_salary.collect()

[('HR', ('Bob', 42000)), ('IT', ('Sam', 75000)), ('Finance', ('Tom', 55000))]

In [87]:
employee_data_reassign = employee_data_rdd.map(lambda x: (x[1],1))
employee_data_reassign.collect()

[('HR', 1),
 ('IT', 1),
 ('HR', 1),
 ('IT', 1),
 ('Finance', 1),
 ('IT', 1),
 ('Finance', 1),
 ('HR', 1)]

In [88]:
employee_total = employee_data_reassign.reduceByKey(lambda a,b: a+b)
employee_total.collect()

[('HR', 3), ('IT', 3), ('Finance', 2)]

In [89]:
employee_max = employee_total.takeOrdered(2,key=lambda x: -x[1])
employee_max

[('HR', 3), ('IT', 3)]

In [90]:
employee_reassign = employee_data_rdd.map(lambda x: (x[1],x[2]))
employee_reassign.collect()

[('HR', 40000),
 ('IT', 60000),
 ('HR', 42000),
 ('IT', 75000),
 ('Finance', 50000),
 ('IT', 62000),
 ('Finance', 55000),
 ('HR', 39000)]

In [91]:
total_payroll = employee_reassign.reduceByKey(lambda a,b: a+b)
total_payroll.collect()

[('HR', 121000), ('IT', 197000), ('Finance', 105000)]

In [92]:
employee_reassign = employee_data_rdd.map(lambda x: (x[1],1))
employee_reassign.collect()

[('HR', 1),
 ('IT', 1),
 ('HR', 1),
 ('IT', 1),
 ('Finance', 1),
 ('IT', 1),
 ('Finance', 1),
 ('HR', 1)]

In [93]:
employees_count = employee_reassign.reduceByKey(lambda a,b: a+b)
employees_count.collect()

[('HR', 3), ('IT', 3), ('Finance', 2)]

In [94]:
employee_reassign = employee_data_rdd.map(lambda x: (x[1],(x[2],1)))
employee_reassign.collect()

[('HR', (40000, 1)),
 ('IT', (60000, 1)),
 ('HR', (42000, 1)),
 ('IT', (75000, 1)),
 ('Finance', (50000, 1)),
 ('IT', (62000, 1)),
 ('Finance', (55000, 1)),
 ('HR', (39000, 1))]

In [95]:
employee_salary = employee_reassign.reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1]))
employee_salary.collect()

[('HR', (121000, 3)), ('IT', (197000, 3)), ('Finance', (105000, 2))]

In [96]:
employee_avg = employee_salary.mapValues(lambda x: x[0]/x[1])
employee_avg.collect()

[('HR', 40333.333333333336), ('IT', 65666.66666666667), ('Finance', 52500.0)]

In [97]:
avg_salary_highest = employee_avg.takeOrdered(1,key=lambda x:-x[1])
avg_salary_highest

[('IT', 65666.66666666667)]

In [98]:
highest_salary_map = employee_data_rdd.map(lambda x:(x[1],(x[0],x[2])))
highest_salary_map.collect()

[('HR', ('John', 40000)),
 ('IT', ('Alice', 60000)),
 ('HR', ('Bob', 42000)),
 ('IT', ('Sam', 75000)),
 ('Finance', ('Eve', 50000)),
 ('IT', ('David', 62000)),
 ('Finance', ('Tom', 55000)),
 ('HR', ('Linda', 39000))]

In [99]:
highest_salary = highest_salary_map.takeOrdered(2,key=lambda x:-x[1][1])
highest_salary

[('IT', ('Sam', 75000)), ('IT', ('David', 62000))]

In [100]:
name = employee_data_rdd.map(lambda x:(x[1],(x[0],len(x[0]))))
name.collect()

[('HR', ('John', 4)),
 ('IT', ('Alice', 5)),
 ('HR', ('Bob', 3)),
 ('IT', ('Sam', 3)),
 ('Finance', ('Eve', 3)),
 ('IT', ('David', 5)),
 ('Finance', ('Tom', 3)),
 ('HR', ('Linda', 5))]

In [101]:
name_length = name.reduceByKey(lambda a,b: a if a[1]>b[1] else b)
name_length.collect()

[('HR', ('Linda', 5)), ('IT', ('David', 5)), ('Finance', ('Tom', 3))]

In [102]:
job_data = [
    ("John", "Senior Data Engineer"),
    ("Alice", "Data Scientist"),
    ("Bob", "Machine Learning Engineer"),
    ("Carol", "Data Analyst"),
    ("David", "Senior Data Scientist"),
    ("Eve", "Business Analyst"),
    ("Frank", "Machine Learning Engineer"),
    ("Grace", "AI Research Scientist"),
    ("Helen", "Junior Data Analyst"),
    ("Irene", "Principal Data Engineer"),
    ("Jack", "Lead Data Scientist"),
    ("Karan", "Big Data Developer"),
    ("Linda", "Junior Business Analyst"),
    ("Mohan", "Data Engineer"),
    ("Nina", "NLP Engineer")
]

In [103]:
job = spark.sparkContext.parallelize(job_data)
job.collect()

[('John', 'Senior Data Engineer'),
 ('Alice', 'Data Scientist'),
 ('Bob', 'Machine Learning Engineer'),
 ('Carol', 'Data Analyst'),
 ('David', 'Senior Data Scientist'),
 ('Eve', 'Business Analyst'),
 ('Frank', 'Machine Learning Engineer'),
 ('Grace', 'AI Research Scientist'),
 ('Helen', 'Junior Data Analyst'),
 ('Irene', 'Principal Data Engineer'),
 ('Jack', 'Lead Data Scientist'),
 ('Karan', 'Big Data Developer'),
 ('Linda', 'Junior Business Analyst'),
 ('Mohan', 'Data Engineer'),
 ('Nina', 'NLP Engineer')]

In [104]:
job_split = job.map(lambda x: x[1].split(' '))
job_split.collect()

[['Senior', 'Data', 'Engineer'],
 ['Data', 'Scientist'],
 ['Machine', 'Learning', 'Engineer'],
 ['Data', 'Analyst'],
 ['Senior', 'Data', 'Scientist'],
 ['Business', 'Analyst'],
 ['Machine', 'Learning', 'Engineer'],
 ['AI', 'Research', 'Scientist'],
 ['Junior', 'Data', 'Analyst'],
 ['Principal', 'Data', 'Engineer'],
 ['Lead', 'Data', 'Scientist'],
 ['Big', 'Data', 'Developer'],
 ['Junior', 'Business', 'Analyst'],
 ['Data', 'Engineer'],
 ['NLP', 'Engineer']]

In [105]:
job_count = job_split.map(lambda x: (x[0],1))
job_count.collect()

[('Senior', 1),
 ('Data', 1),
 ('Machine', 1),
 ('Data', 1),
 ('Senior', 1),
 ('Business', 1),
 ('Machine', 1),
 ('AI', 1),
 ('Junior', 1),
 ('Principal', 1),
 ('Lead', 1),
 ('Big', 1),
 ('Junior', 1),
 ('Data', 1),
 ('NLP', 1)]

In [106]:
jobs = job_count.reduceByKey(lambda a,b: a+b)
jobs.collect()

[('AI', 1),
 ('Lead', 1),
 ('Big', 1),
 ('Senior', 2),
 ('Data', 3),
 ('Machine', 2),
 ('Business', 1),
 ('Junior', 2),
 ('Principal', 1),
 ('NLP', 1)]

In [107]:
frequent_jobs_title = jobs.takeOrdered(1, key = lambda x: -x[1])
frequent_jobs_title

[('Data', 3)]

In [108]:
salary = employee_data_rdd.filter(lambda x: x[2]>50000)
salary.collect()

[('Alice', 'IT', 60000),
 ('Sam', 'IT', 75000),
 ('David', 'IT', 62000),
 ('Tom', 'Finance', 55000)]

In [109]:
def salary_slab(s):
  if s<40000:
    return "low"
  elif s>=40000 and s<=60000:
    return "medium"
  else:
    return "high"

slab_counts  = employee_data_rdd.map(lambda x: (salary_slab(x[2]), 1)).reduceByKey(lambda a, b: a + b).collect()
slab_counts

[('high', 2), ('low', 1), ('medium', 5)]

In [110]:
desc_order_employee= sorted(employee_data,key=lambda x: -x[2])
desc_order_employee

[('Sam', 'IT', 75000),
 ('David', 'IT', 62000),
 ('Alice', 'IT', 60000),
 ('Tom', 'Finance', 55000),
 ('Eve', 'Finance', 50000),
 ('Bob', 'HR', 42000),
 ('John', 'HR', 40000),
 ('Linda', 'HR', 39000)]

In [112]:
lower_salary_list = employee_data_rdd.map(lambda x: (x[1],(x[2])))
lower_salary_list.collect()

[('HR', 40000),
 ('IT', 60000),
 ('HR', 42000),
 ('IT', 75000),
 ('Finance', 50000),
 ('IT', 62000),
 ('Finance', 55000),
 ('HR', 39000)]

In [117]:
lowest_salary = lower_salary_list.reduceByKey(lambda a,b: min(a,b))
lowest_salary.collect()

[('HR', 39000), ('IT', 60000), ('Finance', 50000)]